In [1]:
# 1. Load the Thermal Logic Class (ZoneModel)
%run "One_zone_model_thermal_network_class.ipynb"

# 2. Import required libraries
import pandas as pd
import numpy as np

# --------------------------------------------------------------------------INPUT: SHARED ZONE INPUTS----------------------------------------------------------------------------------------- #

# 1. Define ONLY the base inputs that are truly universal (Building-wide)
base_inputs = {
    # --- Geometry & Location ---
    "t_ground": 15.0,
    "area_roof": 410.0,

    # --- Thermal Properties (Shared Envelope Defaults) ---
    "rc_roof": 5.7,
    "rc_ground_floor": 3.0,
    "u_value_windows": 1.8,
    "alfai": 7.5,
    "alfao": 15.0,
    "thermal_mass_factor": 165000, # Or override it in the individual zones if needed

    # --- Solar Properties ---
    "solar_absorption_coefficient": 0.5,
    "solar_heat_coefficient_shading": 0.5,
    "solar_heat_coefficient_glazing": 0.6,

    # --- Ventilation & Air Properties ---
    "air_density": 1.2,
    "air_heat_capacity": 1003.0,
    "vent_flow_per_person": 36.0,
    "infiltration_ach": 0.2,
    "natural_vent_rate": 3.0,

    # --- Internal Gains ---
    "heat_per_person": 65.0,
    "appliances_w_m2": 6.0,
    "lighting_w_m2": 6.0,

    # --- HVAC System Constants ---
    "heating_power_max": 1000000.0,
    "cooling_power_max": -1000000.0,
    "system_pressure_drop": 200.0,
    "efficiency_fan_and_motor": 0.6,
    
    # Default Schedule fallback
    "system_profile": [1.0]*168
}

# --------------------------------------------------------------------------SHARED INTERNAL WALLS----------------------------------------------------------------------------------------- #

# Define physical properties of shared internal partitions
# Area [m2], R_value [m2K/W]
internal_wall_connections = [
    {"zones": ("Zone 1", "Zone 2"), "area": 15.0, "R": 0.33}, # Shared wall between Zone 1 and Zone 2
    {"zones": ("Zone 2", "Zone 3"), "area": 10.0, "R": 0.33}, # Shared wall between Zone 2 and Zone 3
    {"zones": ("Zone 3", "Zone 4"), "area": 20.0, "R": 0.50}  # etc.
]

# AUTOMATION: Convert physical inputs into UA values for the simulation
adjacencies = []
for connection in internal_wall_connections:
    ua_calculated = connection["area"] / connection["R"]
    adjacencies.append({
        "zones": connection["zones"], 
        "UA": ua_calculated
    })

# Optional: Print the calculated UAs so you can verify them
print("Calculated Internal Wall UAs:")
for adj in adjacencies:
    print(f"  {adj['zones'][0]} <-> {adj['zones'][1]}: {adj['UA']:.2f} W/K")

# --------------------------------------------------------------------------NUMBER OF ZONES AND OWN VARIABLES----------------------------------------------------------------------------------------- #

zone_definitions = []

# --- ZONE 1: NAME 1 ---
z1 = base_inputs.copy()
z1.update({
    "floor_area": 850.0,
    "room_height": 3.0,
    "rc_facade": 3.9,
    "thermal_mass_factor": 1000,
    "area_ground": 100.0,
    "total_facade_areas": {'N': 400.0, 'NE': 436.0, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "window_percentages": {'N': 0.30, 'NE': 0.25, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.80, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.80, 'W': 0.00, 'NW': 0.80},
    "max_people_per_m2": 0.02,
    "vent_cooling_setpoint": 23.9,
    "heating_setpoint": 20.0,
    "cooling_setpoint": 24.0,
    "occ_profile": [0.5]*168,
    "equip_profile": [0.1]*168,
    "vent_profile": [0.5]*168,
})
zone_definitions.append(("Zone 1", z1))

# --- ZONE 2: NAME 2 ---
z2 = base_inputs.copy()
z2.update({
    "floor_area": 450.0,
    "room_height": 3.0,
    "rc_facade": 3.9,
    "thermal_mass_factor": 50000,
    "area_ground": 50.0,
    "total_facade_areas": {'N': 0.0, 'NE': 0.0, 'E': 100.0, 'SE': 390.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "window_percentages": {'N': 0.0, 'NE': 0.0, 'E': 0.30, 'SE': 0.40, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.80, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.80, 'W': 0.00, 'NW': 0.80},
    "max_people_per_m2": 0.10,
    "vent_cooling_setpoint": 23.0,
    "heating_setpoint": 19.0,
    "cooling_setpoint": 24.0,
    "occ_profile": [0.8]*168,
    "equip_profile": [0.3]*168,
    "vent_profile": [0.8]*168,
})
zone_definitions.append(("Zone 2", z2))

# --- ZONE 3: NAME 3 ---
z3 = base_inputs.copy()
z3.update({
    "floor_area": 1200.0,
    "room_height": 3.5, 
    "rc_facade": 4.5, 
    "thermal_mass_factor": 50000,
    "area_ground": 200.0,
    "total_facade_areas": {'N': 0.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "window_percentages": {'N': 0.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.80, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.80, 'W': 0.00, 'NW': 0.80},
    "max_people_per_m2": 0.01,
    "vent_cooling_setpoint": 24.5,
    "heating_setpoint": 18.0,
    "cooling_setpoint": 26.0,
    "occ_profile": [0.2]*168,
    "equip_profile": [0.05]*168,
    "vent_profile": [0.3]*168,
})
zone_definitions.append(("Zone 3", z3))

# --- ZONE 4: NAME 4 ---
z4 = base_inputs.copy()
z4.update({
    "floor_area": 200.0,
    "room_height": 2.8,
    "rc_facade": 3.5,
    "thermal_mass_factor": 50000,
    "area_ground": 60.0,
    "total_facade_areas": {'N': 0.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 150.0, 'W': 100.0, 'NW': 0.0},
    "window_percentages": {'N': 0.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.05, 'W': 0.05, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.80, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.80, 'W': 0.00, 'NW': 0.80},
    "max_people_per_m2": 0.0,
    "vent_cooling_setpoint": 22.0,
    "heating_setpoint": 15.0,
    "cooling_setpoint": 22.0,
    "heating_power_max": 0.0,
    "occ_profile": [0.0]*168,
    "equip_profile": [1.0]*168,
    "vent_profile": [1.0]*168,
})
zone_definitions.append(("Zone 4", z4))

# --------------------------------------------------------------------------RUN ALL----------------------------------------------------------------------------------------- #

file_path = 'Input Temp & Rad - from dynamic excel model.csv'

try:
    weather_df = pd.read_csv(file_path, sep=';')
    weather_df.columns = weather_df.columns.str.strip()
    if 'temp_ext' in weather_df.columns and 'T' not in weather_df.columns:
        weather_df = weather_df.rename(columns={'temp_ext': 'T'})
    
    # 1. Initialize all zone objects and store them in a dictionary
    zones = {}
    for name, params in zone_definitions:
        params['hourly_df'] = weather_df 
        # The __init__ call handles the internal setup
        zones[name] = ZoneModel(name, **params)

    # 2. MASTER HOURLY LOOP (8760 Hours (length of weather dataset))
    for t in range(len(weather_df)):
        zone_external_flows = {name: 0.0 for name in zones.keys()}
        
        # Calculate Heat Exchange via Shared Walls
        for adj in adjacencies:
            z_a_name, z_b_name = adj["zones"]
            ua_value = adj["UA"]
            
            # Grabbing the current state from the class objects
            temp_a = zones[z_a_name].current_temp
            temp_b = zones[z_b_name].current_temp
            
            # Physics: Flow = U * A * Delta_T
            flow_b_to_a = ua_value * (temp_b - temp_a)
            
            # Storing the "push/pull" of heat for each zone
            zone_external_flows[z_a_name] += flow_b_to_a
            zone_external_flows[z_b_name] -= flow_b_to_a 

        # Update each zone's physics
        for name, zone_obj in zones.items():
            zone_obj.calculate_hour_step(t, external_q_flow=zone_external_flows[name])

    # --- NEW: CSV DATA EXPORT BLOCK ---
    hourly_export_data = []
    for t in range(len(weather_df)):
        # Hour index starts at 1 to match your reference file
        row = {"Hour": t + 1}
        for name, zone_obj in zones.items():
            # Combine Heating (+) and Cooling (-) into one 'Demand' value
            # Since they don't occur simultaneously, one will always be 0
            net_demand = zone_obj.annual_heating_results[t] + zone_obj.annual_cooling_results[t]
            row[f"Demand_{name}"] = round(net_demand, 3)
        hourly_export_data.append(row)

    pd.DataFrame(hourly_export_data).to_csv("Results_Yearly_Zone_Demands_Combined.csv", index=False, sep=',')
    # ---------------------------------

    final_results = []
    for name, zone_obj in zones.items():
        # Calculate Totals
        annual_h = sum(zone_obj.annual_heating_results)
        annual_c = sum(zone_obj.annual_cooling_results)
        
        # Calculate Peaks (using max/min from the hourly lists)
        peak_h = max(zone_obj.annual_heating_results) if zone_obj.annual_heating_results else 0.0
        peak_c = min(zone_obj.annual_cooling_results) if zone_obj.annual_cooling_results else 0.0
        
        # Print detailed results per zone (matching Multi_zone_model style)
        print(f"\n--- Results for {name} ---")
        print("Simulation Complete.")
        print(f"Room Volume: {zone_obj.room_volume:.2f} m3")
        print(f"Peak Heating: {peak_h:.2f} kW | Peak Cooling: {peak_c:.2f} kW")
        print(f"Total Annual Heat Demand: {annual_h:.2f} kWh")
        print(f"Total Annual Cool Demand: {annual_c:.2f} kWh")
        
        final_results.append({
            "Zone": name, 
            "Annual Heating [kWh]": round(annual_h, 2), 
            "Annual Cooling [kWh]": round(annual_c, 2)
        })

    # 4. FINAL SUMMARY TABLE
    print("\n" + "="*45)
    print("         MULTI-ZONE SIMULATION RESULTS         ")
    print("="*45)
    print(pd.DataFrame(final_results))
    print("="*45)
    
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Calculated Internal Wall UAs:
  Zone 1 <-> Zone 2: 45.45 W/K
  Zone 2 <-> Zone 3: 30.30 W/K
  Zone 3 <-> Zone 4: 40.00 W/K

--- Results for Zone 1 ---
Simulation Complete.
Room Volume: 2550.00 m3
Peak Heating: 29.13 kW | Peak Cooling: -22.39 kW
Total Annual Heat Demand: 67013.25 kWh
Total Annual Cool Demand: -4135.99 kWh

--- Results for Zone 2 ---
Simulation Complete.
Room Volume: 1350.00 m3
Peak Heating: 26.95 kW | Peak Cooling: -40.77 kW
Total Annual Heat Demand: 42148.93 kWh
Total Annual Cool Demand: -14831.08 kWh

--- Results for Zone 3 ---
Simulation Complete.
Room Volume: 4200.00 m3
Peak Heating: 12.06 kW | Peak Cooling: -5.62 kW
Total Annual Heat Demand: 23491.24 kWh
Total Annual Cool Demand: -11.96 kWh

--- Results for Zone 4 ---
Simulation Complete.
Room Volume: 560.00 m3
Peak Heating: 0.00 kW | Peak Cooling: -11.41 kW
Total Annual Heat Demand: 0.00 kWh
Total Annual Cool Demand: -5499.25 kWh

         MULTI-ZONE SIMULATION RESULTS         
     Zone  Annual Heating [kWh]  Ann